In [30]:
import numpy as np 
from spread_spectrum import SpreadSpectrum

In [31]:
ss = SpreadSpectrum(8, 1024, 0)
ss = SpreadSpectrum(16, 1024, 0)
ss = SpreadSpectrum(32, 1024, 0)
ss

SpreadSpectrum(number=32, length=1024, seed=0)

In [32]:
tx_bytes = np.random.randint(0, 255, size=1 << 12, dtype=np.uint8)
tx_bits = np.unpackbits(tx_bytes).reshape(-1, ss.number)
tx_bips = tx_bits.astype(np.int32) * 2 - 1

tx_bytes[:4], tx_bits[:4], tx_bips[:4]


(array([124, 178, 101, 198], dtype=uint8),
 array([[0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1,
         0, 1, 1, 1, 0, 0, 0, 1, 1, 0],
        [0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0,
         0, 0, 0, 0, 1, 0, 1, 0, 0, 0],
        [0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0,
         1, 1, 0, 1, 1, 0, 0, 1, 0, 1],
        [1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1,
         1, 0, 1, 1, 0, 0, 1, 0, 1, 0]], dtype=uint8),
 array([[-1,  1,  1,  1,  1,  1, -1, -1,  1, -1,  1,  1, -1, -1,  1, -1,
         -1,  1,  1, -1, -1,  1, -1,  1,  1,  1, -1, -1, -1,  1,  1, -1],
        [-1, -1, -1,  1, -1, -1,  1, -1,  1, -1, -1, -1, -1, -1, -1,  1,
         -1,  1,  1,  1, -1, -1, -1, -1, -1, -1,  1, -1,  1, -1, -1, -1],
        [-1, -1,  1,  1,  1,  1, -1, -1, -1,  1,  1, -1,  1, -1, -1,  1,
         -1, -1,  1, -1, -1, -1,  1,  1, -1,  1,  1, -1, -1,  1, -1,  1],
        [ 1, -1, -1,  1, -1,  1,  1, -1, 

In [33]:
sm = ss.spreading_matrix()
sm.shape

(32, 1024)

In [34]:
tx_signal = tx_bips[0] @ sm
tx_signal.shape

(1024,)

In [35]:
ttx_signal = tx_bips @ sm
ttx_signal.shape, ttx_signal.size

((1024, 1024), 1048576)

In [36]:
tx_signal == ttx_signal[0]

array([ True,  True,  True, ...,  True,  True,  True])

In [37]:
rx_bips = sm @ tx_signal 
rx_bips

array([ -974,   878,  1126,   870,  1102,  1114,  -650, -1162,   846,
       -1054,   746,   830, -1006, -1150,  1074, -1182, -1122,   938,
        1274,  -802,  -946,   918, -1078,   986,  1026,   894, -1066,
       -1002,  -970,  1046,   802, -1098])

In [38]:
rrx_bips = ttx_signal @ sm.T
rrx_bips.shape, rrx_bips[0]

((1024, 32),
 array([ -974,   878,  1126,   870,  1102,  1114,  -650, -1162,   846,
        -1054,   746,   830, -1006, -1150,  1074, -1182, -1122,   938,
         1274,  -802,  -946,   918, -1078,   986,  1026,   894, -1066,
        -1002,  -970,  1046,   802, -1098]))

In [39]:
np.all((tx_bips > 0) == (rrx_bips > 0))

True

In [40]:
rrx_bytes = np.packbits(rrx_bips > 0)
rrx_bytes.shape

(4096,)

In [41]:
np.all(rrx_bytes == tx_bytes), ss.length/ss.number*8, np.std(ttx_signal), np.std(rrx_bips)

(True, 256.0, 5.649963292650178, 1035.9628431594685)

In [42]:
ttx_signal.size/len(tx_bytes)

256.0

In [1]:
# all in one cell

for spreading_factor in [128, 256, 512, 1024, 2048, 4096]:
    for bits_per_symbol in [8, 16, 32, 64, 128, 256]:
        ss = SpreadSpectrum(bits_per_symbol, spreading_factor, 0)

        sm = ss.spreading_matrix()

        tx_bytes = np.random.randint(0, 255, size=1 << 16, dtype=np.uint8)
        tx_bits = np.unpackbits(tx_bytes).reshape(-1, ss.number)
        tx_bips = tx_bits.astype(np.int32) * 2 - 1
        ttx_signal = tx_bips @ sm

        rrx_signal = ttx_signal/np.std(ttx_signal)
        rrx_signal += np.random.normal(scale=100, size=ttx_signal.shape)

        rrx_bips = ttx_signal @ sm.T

        rrx_bips_ber = np.mean((tx_bips > 0) != (rrx_bips > 0))
        # print(f"{rrx_bips_ber=}")

        rrx_bytes = np.packbits(rrx_bips > 0)
        rrx_bytes_ber = np.mean(rrx_bytes != tx_bytes)
        # print(f"{rrx_bytes_ber=}")

        print(ss.length, ss.number, ss.length/ss.number*8, rrx_bips_ber)


NameError: name 'SpreadSpectrum' is not defined